In [1]:
import requests 
import yfinance as yf


In [2]:
# Define the ticker symbol
ticker_symbol = "AAPL"

# Create a Ticker object
ticker = yf.Ticker(ticker_symbol)

# Fetch historical market data
historical_data = ticker.history(period="1y")  # data for the last year
print("Historical Data:")
print(historical_data)

# Fetch basic financials
financials = ticker.financials
print("\nFinancials:")
print(financials)

# Fetch stock actions like dividends and splits
actions = ticker.actions
print("\nStock Actions:")
print(actions)

Historical Data:
                                 Open        High         Low       Close  \
Date                                                                        
2024-02-27 00:00:00-05:00  180.248905  183.055644  178.716134  181.771713   
2024-02-28 00:00:00-05:00  181.652272  182.259406  179.283467  180.567398   
2024-02-29 00:00:00-05:00  180.418104  181.711997  178.686276  179.900543   
2024-03-01 00:00:00-05:00  178.706175  179.681565  176.546375  178.815659   
2024-03-04 00:00:00-05:00  175.322137  176.068613  172.973228  174.277084   
...                               ...         ...         ...         ...   
2025-02-20 00:00:00-05:00  244.940002  246.779999  244.289993  245.830002   
2025-02-21 00:00:00-05:00  245.949997  248.690002  245.220001  245.550003   
2025-02-24 00:00:00-05:00  244.929993  248.860001  244.419998  247.100006   
2025-02-25 00:00:00-05:00  248.000000  250.000000  244.910004  247.039993   
2025-02-26 00:00:00-05:00  244.330002  244.979996  239.1300

In [ ]:
options = ticker.options

print(options)

<bound method Ticker.option_chain of yfinance.Ticker object <AAPL>>


In [ ]:

def fetch_option_chain(stock_symbol):
    """Fetch the latest option chain for a given stock"""
    stock = yf.Ticker(stock_symbol)
    exp_dates = stock.options  # List of expiry dates
    options_data = {}

    for exp in exp_dates[:2]:  # Fetch first 2 expiries
        opt_chain = stock.option_chain(exp)
        options_data[exp] = {"calls": opt_chain.calls, "puts": opt_chain.puts}

    return options_data

# Example: Fetch Tesla (TSLA) options
tsla_options = fetch_option_chain("TSLA")
print(tsla_options.keys())  # Expiry dates
print(tsla_options[list(tsla_options.keys())[0]]["calls"].head())  # Call options data


In [ ]:


def fetch_crypto_options(symbol="BTCUSDT"):
    """Fetch crypto options from Binance Futures API"""
    url = "https://fapi.binance.com/fapi/v1/depth"
    params = {"symbol": symbol, "limit": 100}
    response = requests.get(url, params=params).json()
    
    return response  # Returns order book data

# Fetch BTC Futures order book
btc_options_data = fetch_crypto_options("BTCUSDT")
print(btc_options_data["bids"][:5])  # Top 5 bid prices
print(btc_options_data["asks"][:5])  # Top 5 ask prices


In [ ]:
import numpy as np
import scipy.stats as si

def black_scholes(S, K, T, r, sigma, option_type="call"):
    """Black-Scholes Option Pricing Model"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    if option_type == "call":
        price = S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2)
    elif option_type == "put":
        price = K * np.exp(-r * T) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type")
    
    return price


In [ ]:
def calculate_delta(S, K, T, r, sigma, option_type="call"):
    """Calculate Delta (Sensitivity to Price Movements)"""
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    if option_type == "call":
        return si.norm.cdf(d1)
    else:
        return -si.norm.cdf(-d1)

# Example: Delta for Tesla call option
delta = calculate_delta(S=100, K=100, T=0.5, r=0.05, sigma=0.2, option_type="call")
print(f"Call Option Delta: {delta:.4f}")


In [ ]:
def delta_hedge(stock_price, option_strike, time_to_expiry, interest_rate, volatility, option_type):
    """Implement a Delta-Neutral hedging strategy"""
    delta = calculate_delta(stock_price, option_strike, time_to_expiry, interest_rate, volatility, option_type)
    hedge_ratio = -delta  # If delta = 0.5, short 0.5 stock per 1 option
    
    print(f"To hedge: Short {hedge_ratio:.2f} shares per option")
    
    return hedge_ratio

# Example: Hedging a long call option
delta_hedge(stock_price=100, option_strike=100, time_to_expiry=0.5, interest_rate=0.05, volatility=0.2, option_type="call")


In [ ]:
def historical_volatility(prices, window=30):
    """Calculate Historical (Realized) Volatility"""
    log_returns = np.log(prices / prices.shift(1))
    return log_returns.rolling(window=window).std() * np.sqrt(252)

# Example: Calculate realized volatility for TSLA
tsla_prices = yf.download("TSLA", start="2023-01-01", end="2023-12-31")["Close"]
realized_vol = historical_volatility(tsla_prices)


In [ ]:
def volatility_arbitrage(implied_vol, realized_vol):
    """Check if there's an arbitrage opportunity"""
    if implied_vol > realized_vol:
        print("📊 SELL volatility (sell options, short straddles)")
    else:
        print("📊 BUY volatility (buy options, long straddles)")
